# **Multi-Indicator Trading Strategy with Backtesting: RSI, MACD, and Volume Signals on Alphabet Inc. (GOOG)**

# **Goal**

This study aims to develop and evaluate a rule-based trading strategy using multiple technical indicators to determine whether it can outperform a passive buy-and-hold approach.

# **Business Context**

Financial markets are noisy and unpredictable. Traders and quantitative analysts use technical indicators to:

Identify market trends and momentum

Optimize entry and exit timing

Manage risk and capital allocation

The key question:

“Can a structured, indicator-based strategy generate consistent profits compared to simply holding the stock?”

# **Objective**

- Build technical indicators (RSI, MACD, Volume trends)

- Develop a rule-based trading strategy

- Backtest the strategy using historical data

- Compare performance against Buy & Hold

#PROJECT: STOCKS TRADING



##BASIC: Make MA, RSI, Volume, Rolling Stats in Pandas for a Trading Strategy.

##INTERMEDIATE: Implement and Backtest a Trading Strategy.

## Load data

Load historical stock data using yfinance.

In [ ]:
import yfinance as yf
import pandas as pd

# define the stock symbol
ticker_symbol = "GOOG"

# define the start and end dates
start_date = "2000-01-01"
end_date = "2023-01-01"

# get historical data
ticker = yf.Ticker(ticker_symbol)
historical_data = ticker.history(start=start_date, end=end_date)

# display the head rows of the DataFrame
display(historical_data.head())

,Open,High,Low,Close,Volume,Dividends,Stock Splits
Date,,,,,,,
2004-08-19 00:00:00-04:00,2.475947,2.576470,2.375919,2.484366,897427216,0.0,0.0
2004-08-20 00:00:00-04:00,2.500954,2.700763,2.488327,2.681699,458857488,0.0,0.0
2004-08-23 00:00:00-04:00,2.742112,2.809705,2.700021,2.708686,366857939,0.0,0.0
2004-08-24 00:00:00-04:00,2.754243,2.763156,2.564338,2.596526,306396159,0.0,0.0
2004-08-25 00:00:00-04:00,2.598754,2.674023,2.572014,2.624504,184645512,0.0,0.0


## Calculate indicators


Calculate the four specified indicators: Relative Strength Index (RSI), Volume, Volume Moving Average, and Moving Average Convergence/Divergence (MACD).



In [ ]:
# calculate RSI
def calculate_rsi(data, window=14):
    delta = data.diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=window).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=window).mean()
    rs = gain / loss
    rsi = 100 - (100 / (1 + rs))
    return rsi

historical_data['RSI'] = calculate_rsi(historical_data['Close'])

# volume is already available in historical_data

# calculate Volume Moving Average
historical_data['Volume_MA'] = historical_data['Volume'].rolling(window=20).mean()

# calculate MACD
def calculate_macd(data, short_window=12, long_window=26, signal_window=9):
    short_ema = data.ewm(span=short_window, adjust=False).mean()
    long_ema = data.ewm(span=long_window, adjust=False).mean()
    macd_line = short_ema - long_ema
    signal_line = macd_line.ewm(span=signal_window, adjust=False).mean()
    return macd_line, signal_line

historical_data['MACD'], historical_data['MACD_Signal'] = calculate_macd(historical_data['Close'])

# handle nans by dropping rows
historical_data.dropna(inplace=True)

# display the head rows with the new indicators
display(historical_data.head())

,Open,High,Low,Close,Volume,Dividends,Stock Splits,RSI,Volume_MA,MACD,MACD_Signal
Date,,,,,,,,,,,
2004-09-16 00:00:00-04:00,2.781478,2.867146,2.764395,2.821836,186207345,0.0,0.0,62.071651,2.263332e+08,0.056587,0.029000
2004-09-17 00:00:00-04:00,2.832978,2.908990,2.811437,2.908990,190350817,0.0,0.0,71.109428,1.909794e+08,0.072859,0.037772
2004-09-20 00:00:00-04:00,2.895620,3.010752,2.891164,2.955291,213585582,0.0,0.0,85.278506,1.787158e+08,0.088471,0.047912
2004-09-21 00:00:00-04:00,2.966432,2.981535,2.909485,2.917656,145262446,0.0,0.0,80.038727,1.676360e+08,0.096693,0.057668
2004-09-22 00:00:00-04:00,2.906762,2.962965,2.892153,2.931025,152344894,0.0,0.0,87.505045,1.599334e+08,0.103098,0.066754


## Develop trading strategy

Implement the trading strategy logic, create the 'Signal' column, populate it based on the conditions, and display the head of the dataframe.



In [ ]:
# define the trading strategy logic
# buy when MACD crosses above MACD Signal and RSI is below 40
# sell when MACD crosses below MACD Signal and RSI is above 60

historical_data['Signal'] = None

for i in range(1, len(historical_data)):
    # buy signal
    if (historical_data['MACD'].iloc[i] > historical_data['MACD_Signal'].iloc[i]) and \
       (historical_data['MACD'].iloc[i-1] <= historical_data['MACD_Signal'].iloc[i-1]) and \
       (historical_data['RSI'].iloc[i] < 40):
        historical_data.loc[historical_data.index[i], 'Signal'] = 'Buy'

    # sell signal
    elif (historical_data['MACD'].iloc[i] < historical_data['MACD_Signal'].iloc[i]) and \
         (historical_data['MACD'].iloc[i-1] >= historical_data['MACD_Signal'].iloc[i-1]) and \
         (historical_data['RSI'].iloc[i] > 60):
        historical_data.loc[historical_data.index[i], 'Signal'] = 'Sell'

# display the head of the DataFrame with the new 'Signal' column
display(historical_data.head())

,Open,High,Low,Close,Volume,Dividends,Stock Splits,RSI,Volume_MA,MACD,MACD_Signal,Signal
Date,,,,,,,,,,,,
2004-09-16 00:00:00-04:00,2.781478,2.867146,2.764395,2.821836,186207345,0.0,0.0,62.071651,2.263332e+08,0.056587,0.029000,None
2004-09-17 00:00:00-04:00,2.832978,2.908990,2.811437,2.908990,190350817,0.0,0.0,71.109428,1.909794e+08,0.072859,0.037772,None
2004-09-20 00:00:00-04:00,2.895620,3.010752,2.891164,2.955291,213585582,0.0,0.0,85.278506,1.787158e+08,0.088471,0.047912,None
2004-09-21 00:00:00-04:00,2.966432,2.981535,2.909485,2.917656,145262446,0.0,0.0,80.038727,1.676360e+08,0.096693,0.057668,None
2004-09-22 00:00:00-04:00,2.906762,2.962965,2.892153,2.931025,152344894,0.0,0.0,87.505045,1.599334e+08,0.103098,0.066754,None


## Backtest strategy

Implement the backtesting logic by iterating through the historical data, simulating trades based on the 'Signal' column, tracking capital, position, and trade details.



In [ ]:
# initialize variables for backtesting
starting_capital = 100000
current_capital = starting_capital
shares_held = 0
in_position = False
buy_price = 0
trade_log = [] # to store details of each trade
position_size_percentage = 0.1 # percentage of capital to use for each trade

# create a DataFrame to store trade logs
trade_log_df = pd.DataFrame(columns=['Buy_Date', 'Buy_Price', 'Sell_Date', 'Sell_Price', 'Shares', 'Profit_Loss'])


# iterate through the historical data to simulate trades
for i in range(1, len(historical_data)):
    signal = historical_data['Signal'].iloc[i]
    close_price = historical_data['Close'].iloc[i]
    current_date = historical_data.index[i]

    # buy signal
    if signal == 'Buy' and not in_position:
        investment_amount = current_capital * position_size_percentage
        num_shares_to_buy = investment_amount // close_price
        if num_shares_to_buy > 0:
            shares_held = num_shares_to_buy
            buy_price = close_price
            in_position = True
            current_capital -= shares_held * buy_price
            trade_log.append({'Buy_Date': current_date, 'Buy_Price': buy_price, 'Sell_Date': None, 'Sell_Price': None, 'Shares': shares_held, 'Profit_Loss': None})

    # sell signal
    elif signal == 'Sell' and in_position:
        sell_price = close_price
        profit_loss = (sell_price - buy_price) * shares_held
        current_capital += shares_held * sell_price

        # find the corresponding buy trade in trade_log and update it
        for trade in reversed(trade_log):
            if trade['Sell_Date'] is None:
                trade['Sell_Date'] = current_date
                trade['Sell_Price'] = sell_price
                trade['Profit_Loss'] = profit_loss
                break

        in_position = False
        shares_held = 0
        buy_price = 0

# if still in a position at the end of the data, close the position
if in_position:
    sell_price = historical_data['Close'].iloc[-1]
    profit_loss = (sell_price - buy_price) * shares_held
    current_capital += shares_held * sell_price

    # find the corresponding buy trade in trade_log and update it
    for trade in reversed(trade_log):
        if trade['Sell_Date'] is None:
            trade['Sell_Date'] = historical_data.index[-1]
            trade['Sell_Price'] = sell_price
            trade['Profit_Loss'] = profit_loss
            break

# convert trade_log list to DataFrame
trade_log_df = pd.DataFrame(trade_log)

# 1. Calculate the total profit/loss from the trade_log_df.
total_profit_loss = trade_log_df['Profit_Loss'].sum()

# 2. Calculate the number of winning trades and losing trades from the trade_log_df.
winning_trades = trade_log_df[trade_log_df['Profit_Loss'] > 0]
losing_trades = trade_log_df[trade_log_df['Profit_Loss'] <= 0] # consider trades with 0 profit as losing or break-even, depending on definition

num_winning_trades = len(winning_trades)
num_losing_trades = len(losing_trades)
total_trades = num_winning_trades + num_losing_trades

# 3. Calculate the win rate
win_rate = num_winning_trades / total_trades if total_trades > 0 else 0

# 4. Calculate the average profit per winning trade and the average loss per losing trade.
average_profit_per_win = winning_trades['Profit_Loss'].mean() if num_winning_trades > 0 else 0
average_loss_per_loss = losing_trades['Profit_Loss'].mean() if num_losing_trades > 0 else 0

# 5. Calculate the maximum drawdown from the historical_data and the capital values during the backtest.
# assuming backtest started with starting_capital and trades happen at close price
starting_capital_drawdown = 100000 # Use a separate variable for drawdown calculation
capital_progression = [starting_capital_drawdown]
current_capital_drawdown = starting_capital_drawdown
shares_held_drawdown = 0
buy_price_drawdown = 0
in_position_drawdown = False


for i in range(1, len(historical_data)):
    signal = historical_data['Signal'].iloc[i]
    close_price = historical_data['Close'].iloc[i]

    if signal == 'Buy' and not in_position_drawdown:
        investment_amount = current_capital_drawdown * position_size_percentage
        num_shares_to_buy = investment_amount // close_price
        if num_shares_to_buy > 0:
            shares_held_drawdown = num_shares_to_buy
            buy_price_drawdown = close_price
            in_position_drawdown = True
            current_capital_drawdown -= shares_held_drawdown * buy_price_drawdown

    elif signal == 'Sell' and in_position_drawdown:
        sell_price = close_price
        current_capital_drawdown += shares_held_drawdown * sell_price
        in_position_drawdown = False
        shares_held_drawdown = 0
        buy_price_drawdown = 0

    # if in position, the current value of the position contributes to the total capital
    current_portfolio_value = current_capital_drawdown + (shares_held_drawdown * close_price if in_position_drawdown else 0)
    capital_progression.append(current_portfolio_value)

# if still in a position at the end, add the final liquidation value
if in_position_drawdown:
  current_capital_drawdown += shares_held_drawdown * historical_data['Close'].iloc[-1]
  capital_progression.append(current_capital_drawdown)


capital_progression_series = pd.Series(capital_progression)
peak_capital = capital_progression_series.cummax()
drawdown = (peak_capital - capital_progression_series) / peak_capital
max_drawdown = drawdown.max()


# display the trade log DataFrame
display(trade_log_df)

# 6. Print or display the calculated metrics
print("Backtesting Results:")
print(f"Starting Capital: {starting_capital:.2f}")
print(f"Ending Capital (Strategy): {current_capital:.2f}")
print(f"Total Profit/Loss (Strategy): {total_profit_loss:.2f}")
print(f"Number of Winning Trades: {num_winning_trades}")
print(f"Number of Losing Trades: {num_losing_trades}")
print(f"Total Trades: {total_trades}")
print(f"Win Rate: {win_rate:.2%}")
print(f"Average Profit per Winning Trade: {average_profit_per_win:.2f}")
print(f"Average Loss per Losing Trade: {average_loss_per_loss:.2f}")
print(f"Maximum Drawdown (Strategy): {max_drawdown:.2%}")

,Buy_Date,Buy_Price,Sell_Date,Sell_Price,Shares,Profit_Loss
0,2005-03-23 00:00:00-05:00,4.431449,2005-05-13 00:00:00-04:00,5.675860,2256.0,2807.391197
1,2006-02-22 00:00:00-05:00,9.049339,2006-09-29 00:00:00-04:00,9.950831,1136.0,1024.095047
2,2007-03-09 00:00:00-05:00,11.215050,2007-06-14 00:00:00-04:00,12.450053,925.0,1142.378211
3,2008-03-12 00:00:00-04:00,10.898623,2008-05-13 00:00:00-04:00,14.434771,963.0,3405.310593
4,2008-08-06 00:00:00-04:00,12.041520,2009-04-28 00:00:00-04:00,9.500456,900.0,-2286.957836
5,2010-02-11 00:00:00-05:00,13.280980,2010-11-01 00:00:00-04:00,15.227074,798.0,1552.982660
6,2011-03-23 00:00:00-04:00,14.413974,2011-08-02 00:00:00-04:00,14.667510,746.0,189.138023
7,2014-04-30 00:00:00-04:00,26.105724,2014-07-10 00:00:00-04:00,28.308546,413.0,909.765375
8,2014-10-24 00:00:00-04:00,26.756063,2015-08-03 00:00:00-04:00,31.374004,406.0,1874.884007
9,2016-02-19 00:00:00-05:00,34.838417,2016-08-16 00:00:00-04:00,38.627388,317.0,1201.103790


Backtesting Results:
Starting Capital: 100000.00
Ending Capital (Strategy): 115396.69
Total Profit/Loss (Strategy): 15396.69
Number of Winning Trades: 11
Number of Losing Trades: 1
Total Trades: 12
Win Rate: 91.67%
Average Profit per Winning Trade: 1607.60
Average Loss per Losing Trade: -2286.96
Maximum Drawdown (Strategy): 5.17%


## Buy and Hold strategy

Implement a buy-and-hold strategy by buying on the first available date and selling on the most recent date.

In [ ]:
# 1. Calculate buy and hold profit/loss
# get the first and last closing prices
initial_price = historical_data['Close'].iloc[0]
final_price = historical_data['Close'].iloc[-1]

# calculate the percentage change
buy_and_hold_return_percentage = (final_price - initial_price) / initial_price

# calculate the buy and hold profit/loss in dollar terms
starting_capital = 100000
buy_and_hold_profit_loss = starting_capital * buy_and_hold_return_percentage

# display the results of the buy and hold strategy
print("Buy and Hold Results:")
print(f"Starting Capital: {starting_capital:.2f}")
print(f"Ending Capital (Buy and Hold): {starting_capital + buy_and_hold_profit_loss:.2f}")
print(f"Total Profit/Loss (Buy and Hold): {buy_and_hold_profit_loss:.2f}")

Buy and Hold Results:
Starting Capital: 100000.00
Ending Capital (Buy and Hold): 3125825.92
Total Profit/Loss (Buy and Hold): 3025825.92


## Compare results


Compare the results of backtesting strategy and buy hold strategy

In [ ]:
# compare the two strategies
if total_profit_loss > buy_and_hold_profit_loss:
    print("\nThe trading strategy outperformed the buy and hold strategy in terms of total profit/loss.")
elif total_profit_loss < buy_and_hold_profit_loss:
    print("\nThe buy and hold strategy outperformed the trading strategy in terms of total profit/loss.")
else:
    print("\nThe trading strategy and buy and hold strategy had the same total profit/loss.")


The buy and hold strategy outperformed the trading strategy in terms of total profit/loss.


## Final Thoughts:

### Key Findings from the Backtest

* The test ran on Google (GOOG) stock, starting from Sept 16, 2004 up until Dec 30, 2022.
* In total, the strategy only fired off 12 trades, which isn’t a lot for almost 20 years.
* Starting with USD100,000, it ended up making about +USD15,397 profit (so not huge).
* The win rate though was surprisingly high — 11 winners out of 12 trades, that’s ~91.7%.
* Winning trades on average made around +USD1,608, while the single loser cost about -USD2,287.
* The max drawdown was 5.17%, which is pretty low, so downside wasn’t too scary here.

### Strategy Comparison

* Strategy total profit: about +USD15,397.
* Buy & hold over the exact same time: roughly +USD3,025,826
* So yeah, buy and hold crushed it, no way around that.

### Takeaways

* The system looks good on paper for win rate + risk control, but with only 12 trades the sample size is tiny. The profit is also nowhere near buy & hold.
* Would probably help to test on different time periods (bull, bear, sideways markets) and see if it shines in certain conditions.
* Digging into each trade (entries, exits, etc.) might give some clues about when it works well and when it just doesn’t.
* Could be worth tweaking the indicators/parameters, or adding some filters to avoid bad setups and hopefully push profits higher.